# 08. Proyek 2: implementasi attention dan Transformer

Menghitung scaled dot-product attention, membandingkannya dengan implementasi PyTorch, menambahkan posisi serta padding mask, dan melatih Transformer encoder kecil. Modul ini adalah implementasi mekanisme, bukan replikasi hasil eksperimen sebuah makalah.

**Prasyarat:** modul 03 dan 07.

**Pola belajar:** baca penjelasan, prediksi bentuk keluaran, jalankan kode, lalu ubah satu hal.

Contoh ulasan dalam paket ini merupakan data sintetis untuk mempelajari mekanisme. Metriknya tidak mewakili kinerja pada ulasan nyata.

## Penyiapan

Instal dependensi melalui petunjuk README sebelum menjalankan seluruh sel. Setiap notebook dapat dimulai dengan kernel baru. GPU bersifat opsional. Semua operasi tensor yang berinteraksi harus berada pada perangkat yang sesuai.

In [1]:
from pathlib import Path
import sys
# Lokal: buka dari root repo, folder nlp, atau nlp/notebooks.
# Colab: ambil paket kursus jika belum tersedia.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "nlp")
             if (p / "nlp_course").is_dir()), None)
if ROOT is None and "google.colab" in sys.modules:
    import subprocess
    target = Path("/content/pytorch-deep-learning-nlp")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", "--branch", "nlp-learning-path",
                        "https://github.com/FeliksMakarios/pytorch-deep-learning.git",
                        str(target)], check=True)
        subprocess.run(["git", "sparse-checkout", "set", "nlp"], cwd=target, check=True)
    ROOT = target / "nlp"
if ROOT is None or not (ROOT / "nlp_course").is_dir():
    raise RuntimeError("Folder nlp_course tidak ditemukan. Ikuti petunjuk README nlp.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from torch import nn
from nlp_course.data import tokenize, build_vocab, encode, read_rows, loaders, collate_batch
from nlp_course.models import MeanClassifier, RecurrentClassifier, TinyTransformer
from nlp_course.engine import seed_all, fit, run_epoch, metrics, save_mean, load_mean, predict
seed_all(42)
torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "Perangkat:", device)

PyTorch: 2.14.0+cu130 Perangkat: cpu


## 1. Query, key, dan value

Setiap token diproyeksikan menjadi query, key, dan value. Skor kemiripan query-key menentukan campuran value. Untuk satu head, attention = softmax(QKᵀ / √d_k)V. Softmax bekerja pada sumbu token key.

In [2]:
import math
seed_all(42)
x = torch.randn(2,4,8)
wq,wk,wv = [nn.Linear(8,8,bias=False) for _ in range(3)]
q,k,v = wq(x),wk(x),wv(x)
scores = q @ k.transpose(-2,-1) / math.sqrt(q.shape[-1])
weights = scores.softmax(-1)
output = weights @ v
print("Q/K/V:",q.shape,"Skor:",scores.shape,"Output:",output.shape)
assert torch.allclose(weights.sum(-1),torch.ones(2,4))

Q/K/V: torch.Size([2, 4, 8]) Skor: torch.Size([2, 4, 4]) Output: torch.Size([2, 4, 8])


## 2. Mask untuk padding

Token PAD tidak boleh menjadi sumber informasi attention. Di perhitungan manual ini, True pada pad_mask berarti posisi diblokir. Setelah attention, posisi query PAD juga dikeluarkan dari pooling. Pastikan setiap urutan memiliki sedikitnya satu token nyata.

In [3]:
pad_mask = torch.tensor([[False,False,False,True],[False,False,True,True]])
masked_scores = scores.masked_fill(pad_mask[:,None,:],float("-inf"))
weights = masked_scores.softmax(-1)
manual = weights @ v
print(weights[1])
assert torch.all(weights[1,:,2:] == 0)

tensor([[0.5005, 0.4995, 0.0000, 0.0000],
        [0.4758, 0.5242, 0.0000, 0.0000],
        [0.4898, 0.5102, 0.0000, 0.0000],
        [0.5758, 0.4242, 0.0000, 0.0000]], grad_fn=<SelectBackward0>)


## 3. Memeriksa hasil terhadap PyTorch

Konvensi mask perlu diperiksa per API. Pada scaled_dot_product_attention, boolean True berarti posisi diizinkan. Pada TransformerEncoder, src_key_padding_mask True berarti PAD diblokir. Perbedaan ini sering menjadi sumber kesalahan.

In [4]:
from torch.nn.functional import scaled_dot_product_attention
# Tambahkan dimensi head: [B,H,T,D]. True berarti diizinkan pada API ini.
allowed = (~pad_mask)[:,None,None,:]
reference = scaled_dot_product_attention(q[:,None],k[:,None],v[:,None],
                                         attn_mask=allowed,dropout_p=0.0).squeeze(1)
print("Selisih maksimum:",(manual-reference).abs().max().item())
assert torch.allclose(manual,reference,atol=1e-5)

Selisih maksimum: 5.960464477539063e-08


## 4. Posisi, residual, dan feed-forward

Self-attention tanpa posisi tidak menyatakan letak token secara eksplisit. Model kursus menambahkan embedding posisi yang dipelajari. TransformerEncoderLayer juga memiliki multi-head attention, koneksi residual, normalisasi, dan jaringan feed-forward.

Classifier ini membaca seluruh kalimat sehingga tidak memakai causal mask. Model pembangkit autoregresif memerlukan causal mask agar posisi saat ini tidak melihat token masa depan.

In [5]:
import inspect
print(inspect.getsource(TinyTransformer))

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, dim=32, heads=4, classes=2, max_length=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.position = nn.Embedding(max_length, dim)
        layer = nn.TransformerEncoderLayer(dim, heads, dim_feedforward=dim*2,
                                           dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=1, enable_nested_tensor=False)
        self.head = nn.Linear(dim, classes)
        self.scale = math.sqrt(dim)
    def forward(self, ids, lengths):
        if ids.shape[1] > self.position.num_embeddings:
            raise ValueError('Urutan melampaui kapasitas posisi model')
        pos = torch.arange(ids.shape[1], device=ids.device)
        x = self.embedding(ids) * self.scale + self.position(pos)
        x = self.encoder(x, src_key_padding_mask=ids.eq(0))
        mask = ids.ne(0).unsqueeze(-1)
        return self.

## 5. Melatih Transformer kecil

Transformer ini dilatih dari awal. Ukuran kecil dan data sintetis tidak setara dengan BERT. Gunakan model ini untuk memahami aliran tensor dan mask, lalu bandingkan dengan model sederhana melalui validasi.

In [6]:
vocab,train,val,test = loaders()
seed_all(42)
model = TinyTransformer(len(vocab),dim=16,heads=2)
history = fit(model,train,val,epochs=15,lr=0.003)
print(history[-1])
print("Validasi:",run_epoch(model,val))
print("Uji checkpoint terpilih:",run_epoch(model,test))

{'epoch': 15, 'train_loss': 0.0366703228921526, 'val_loss': 1.204203337430954, 'val_macro_f1': 0.644444465637207}
Validasi: {'loss': 0.5801661113897959, 'accuracy': 0.6666666865348816, 'macro_f1': 0.6630101203918457, 'confusion': [[27, 21], [11, 37]]}
Uji checkpoint terpilih: {'loss': 0.5154228508472443, 'accuracy': 0.7395833134651184, 'macro_f1': 0.7393287420272827, 'confusion': [[37, 11], [14, 34]]}


## 6. Menguji posisi PAD

Setelah dropout dinonaktifkan melalui eval(), menambah PAD di ujung tidak boleh mengubah prediksi secara berarti. Toleransi numerik diperlukan untuk operasi floating point.

In [7]:
model.eval()
ids,lengths,_=next(iter(val))
with torch.inference_mode():
    base=model(ids,lengths)
    extended=model(torch.nn.functional.pad(ids,(0,3)),lengths)
print("Selisih padding:",(base-extended).abs().max().item())
assert torch.allclose(base,extended,atol=1e-5)

Selisih padding: 1.1920928955078125e-07


## Latihan mandiri

1. Mengapa skor dibagi akar dimensi key?
2. Apa arti True pada dua jenis mask di atas?
3. Mengapa classifier tidak memakai causal mask?
4. Apakah attention weight membuktikan penjelasan sebab-akibat model?

## Pembahasan latihan

1. Skala tersebut membantu mengendalikan besarnya dot product saat dimensi bertambah.
2. True memblokir pada src_key_padding_mask, tetapi mengizinkan pada boolean attn_mask scaled_dot_product_attention.
3. Semua token kalimat sudah tersedia saat klasifikasi.
4. Tidak. Bobot attention memperlihatkan suatu operasi internal dan perlu analisis tambahan untuk klaim interpretasi.

## Penghubung ke materi berikutnya

Modul 09 memakai checkpoint classifier untuk membuat antarmuka prediksi dan menguji konsistensi prapemrosesan.

### Rujukan
- [Dokumentasi PyTorch](https://docs.pytorch.org/docs/stable/index.html)
- [Sumber Embedding](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/sparse.py)
- [Sumber Transformer](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/transformer.py)
- [Kursus sumber dan struktur awal](https://github.com/mrdbourke/pytorch-deep-learning)

Materi ini ditulis sebagai jalur NLP mandiri. Penjelasan dan contoh NLP bukan terjemahan resmi kursus sumber.